# Module 9 — Debugging challenge

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/python_tutorial/09_debugging_challenge.ipynb)
&nbsp; Part of the **Python for biosignal processing** tutorial — see the [overview](README.md).

This notebook is broken on purpose. It contains **10 challenges**, each with **exactly one bug**.
They are the mistakes from Module 8, hidden in code that looks reasonable — which is how they appear in real
projects.

**How to work through a challenge**

1. Read the **goal** and the **symptom**.
2. Run the challenge cell and **reproduce** the symptom.
3. **Investigate:** print shapes, types and a few values; plot; compare with a number you know. Stuck? Open
   the hints one at a time.
4. **Fix the bug** in the challenge cell. Change only what is wrong.
5. Run the **check** cell. It checks each part of the goal separately and prints `PASS` or `NOT YET` for each.
6. **Explain your fix** in two lines, in the cell below the check.

**Rules of the game:** run the setup cell first. Every challenge makes its own data, so you can do them in any
order. Challenge 1 stops with an error on purpose, so run the notebook **cell by cell**, not with *Run all*.
The stars show the difficulty (★ to ★★★).

When you have tried — really tried — the [worked troubleshooting steps](09_debugging_challenge_solutions.ipynb) show how to find each bug,
not just the answer.

## Setup — run this first

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

fs = 250                                          # Hz, used by most challenges

def make_ecg(seconds=10, hr_bpm=72, fs=250, seed=0):
    """A synthetic ECG-like trace in mV: R and T waves, slow baseline wander and a little noise."""
    r = np.random.default_rng(seed)
    t = np.arange(int(seconds * fs)) / fs
    x = np.zeros(t.size)
    for beat in np.arange(0.3, seconds, 60 / hr_bpm):
        x += 1.00 * np.exp(-((t - beat) ** 2) / (2 * 0.012 ** 2))          # R wave
        x += 0.25 * np.exp(-((t - beat - 0.25) ** 2) / (2 * 0.04 ** 2))    # T wave
    x += 0.3 * np.sin(2 * np.pi * 0.25 * t)                               # baseline wander
    return x + 0.03 * r.standard_normal(t.size)

results = {}          # challenge number -> {requirement: True/False}, or None if its code didn't finish
current = None        # set by each check cell

def check(ok, what):
    """Print PASS / NOT YET for one requirement of the current challenge, and record it."""
    results.setdefault(current, {})[what] = bool(ok)
    print(("PASS     " if ok else "NOT YET  ") + what)

## Challenge 1 — The sample count ★

**Goal:** Compute how many samples a 10-second recording at `fs` has.

**Symptom:** The cell stops with an error.

In [ ]:
duration_s = 10
n_samples = int(duration_s * sampling_rate)
print(n_samples)

**Stuck?** Open one hint at a time.

<details><summary>Hint 1</summary>

Read the last line of the error message: which name does Python not know?
</details>

<details><summary>Hint 2</summary>

Which names does the setup cell define? (`dir()` lists every defined name.)
</details>

<details><summary>Hint 3</summary>

The sampling rate already has a name in this notebook.
</details>

**Check** (don't edit):

In [ ]:
current = 1
results[1] = {}
try:
    check(n_samples == 2500, "a 10-s recording at 250 Hz has 2500 samples")
except NameError:
    results[1] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Explain your fix** (double-click to edit):

- The wrong assumption was: …
- The evidence that showed it: …
- A check that would catch it again: …

## Challenge 2 — The time axis ★

**Goal:** Plot the ECG against time in seconds. The recording lasts 10 s, so its last sample is at (N − 1)/fs = 9.996 s.

**Symptom:** The plot's x-axis runs to more than 600,000 seconds.

**Predict first:** before running, write down what `t[1]` and `t[-1]` should be.

In [ ]:
ecg = make_ecg(fs=fs)
t = np.arange(ecg.size) * fs

fig, ax = plt.subplots(figsize=(9, 2.5))
ax.plot(t, ecg); ax.set_xlabel("time (s)"); ax.set_ylabel("amplitude (mV)")
plt.show()

**Stuck?** Open one hint at a time.

<details><summary>Hint 1</summary>

Print the first three values of `t` and the last one. What should they be?
</details>

<details><summary>Hint 2</summary>

How long is one sampling interval at 250 Hz?
</details>

<details><summary>Hint 3</summary>

Sample number n happens at time n × (one sampling interval).
</details>

**Check** (don't edit):

In [ ]:
current = 2
results[2] = {}
try:
    check(t.size == ecg.size and t[0] == 0, "one time value per sample, starting at 0 s")
    check(np.allclose(np.diff(t), 1 / fs), "steps of 1/fs = 0.004 s")
    check(np.isclose(t[-1], (ecg.size - 1) / fs), "last sample at (N - 1)/fs = 9.996 s")
except NameError:
    results[2] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Explain your fix** (double-click to edit):

- The wrong assumption was: …
- The evidence that showed it: …
- A check that would catch it again: …

## Challenge 3 — Per-channel RMS ★★

**Goal:** A two-channel recording arrives the way WFDB returns it, as `(n_samples, n_channels)`. Compute the RMS (root mean square) of each channel: two numbers.

**Symptom:** No error, but you get 2500 numbers instead of 2.

**Predict first:** what shape should the result have, and which channel should have the larger RMS?

In [ ]:
rec = np.c_[make_ecg(seed=1), 0.5 * make_ecg(seed=2)]       # (n_samples, n_channels), like WFDB

def rms_per_channel(x):
    """RMS of each channel. x must be (n_channels, n_samples)."""
    return np.sqrt(np.mean(x ** 2, axis=1))

rms = rms_per_channel(rec)
print(rms.shape)

**Stuck?** Open one hint at a time.

<details><summary>Hint 1</summary>

Print `rec.shape`. Then read the function's docstring: what shape does it expect?
</details>

<details><summary>Hint 2</summary>

With `axis=1`, which dimension does `np.mean` average over, for an array shaped like `rec`?
</details>

<details><summary>Hint 3</summary>

There are two ways to agree on the layout: change the data you pass, or the axis the function uses.
</details>

**Check** (don't edit):

In [ ]:
current = 3
results[3] = {}
try:
    reference = np.array([np.sqrt(np.mean(rec[:, k] ** 2)) for k in range(2)])   # RMS, one channel at a time
    check(np.shape(rms) == (2,), "one value per channel")
    check(np.shape(rms) == (2,) and np.allclose(rms, reference), "the values are the RMS of each channel")
except NameError:
    results[3] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Explain your fix** (double-click to edit):

- The wrong assumption was: …
- The evidence that showed it: …
- A check that would catch it again: …

## Challenge 4 — Scaling ADC counts ★★

**Goal:** The amplifier stored the ECG as 16-bit integers (`int16`). Each count is 30 µV. Convert the whole recording to **floating-point** values in µV, the representation all later processing expects. (The numbers are exaggerated so the problem is easy to see.)

**Symptom:** The R peaks look clipped, and some samples jump to large negative values.

**Predict first:** roughly what is the largest value in µV, if the largest count is 1304?

In [ ]:
raw = (make_ecg(seed=3) * 1000).astype(np.int16)             # ADC counts, as stored in the file
gain_uv_per_count = 30
ecg_uv = raw * gain_uv_per_count
print(ecg_uv.dtype, ecg_uv.min(), ecg_uv.max())

**Stuck?** Open one hint at a time.

<details><summary>Hint 1</summary>

Print `ecg_uv.dtype`, `ecg_uv.max()` and `raw.max() * 30` computed by hand (1304 × 30).
</details>

<details><summary>Hint 2</summary>

What is the largest number an `int16` can hold?
</details>

<details><summary>Hint 3</summary>

Change the type *before* you multiply.
</details>

**Check** (don't edit):

In [ ]:
current = 4
results[4] = {}
try:
    expected = raw.astype(np.float64) * 30                       # a safe reference, every sample
    check(np.issubdtype(ecg_uv.dtype, np.floating), "floating-point result")
    check(np.shape(ecg_uv) == raw.shape and np.allclose(ecg_uv, expected), "every sample equals counts × 30 µV")
except NameError:
    results[4] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Explain your fix** (double-click to edit):

- The wrong assumption was: …
- The evidence that showed it: …
- A check that would catch it again: …

## Challenge 5 — Removing baseline wander ★★

**Goal:** Remove the slow baseline wander (below 0.5 Hz) with a high-pass filter, keeping the R peaks (about 1 mV).

**Symptom:** The wander is gone — but so are the R peaks: the largest value is now about 0.07.

In [ ]:
ecg5 = make_ecg(seed=4)
sos = signal.butter(2, 0.5, btype="highpass", output="sos")      # remove everything below 0.5 Hz
ecg5_hp = signal.sosfiltfilt(sos, ecg5)
print(f"max before {ecg5.max():.2f}, after {ecg5_hp.max():.2f}")

**Stuck?** Open one hint at a time.

<details><summary>Hint 1</summary>

Don't only look at the output: where does this filter actually cut off? Plot `signal.sosfreqz(sos, fs=fs)`.
</details>

<details><summary>Hint 2</summary>

What does `signal.butter` assume the number 0.5 means when you don't tell it the sampling rate?
</details>

<details><summary>Hint 3</summary>

Every filter-design call should know `fs`.
</details>

**Check** (don't edit):

In [ ]:
current = 5
results[5] = {}
try:
    f_chk, P_before = signal.welch(ecg5, fs=fs, nperseg=8 * fs)
    _, P_after = signal.welch(ecg5_hp, fs=fs, nperseg=8 * fs)
    low = f_chk <= 0.3
    check(P_after[low].sum() < 0.1 * P_before[low].sum(), "the slow wander is strongly reduced")
    check(ecg5_hp.max() > 0.8, "the R peaks are kept (maximum above 0.8 mV)")
except NameError:
    results[5] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Explain your fix** (double-click to edit):

- The wrong assumption was: …
- The evidence that showed it: …
- A check that would catch it again: …

## Challenge 6 — Heart rate from R peaks ★★

**Goal:** Find the R peaks and compute the heart rate in beats per minute. The true rate is 72 bpm.

**Symptom:** The heart rate comes out as 0.29 bpm.

In [ ]:
ecg6 = make_ecg(seed=5)
ecg6 = signal.sosfiltfilt(signal.butter(2, 0.5, btype="highpass", fs=fs, output="sos"), ecg6)  # correct
peaks, _ = signal.find_peaks(ecg6, height=0.5, distance=int(0.3 * fs))
rr = np.diff(peaks)                                   # RR intervals
hr_bpm = 60 / np.mean(rr)
print(f"{peaks.size} beats, HR = {hr_bpm:.2f} bpm")

**Stuck?** Open one hint at a time.

<details><summary>Hint 1</summary>

Is the number of beats plausible for 10 s at about 72 bpm? Then look *after* the peak finding.
</details>

<details><summary>Hint 2</summary>

Print `rr[:3]`. In what unit are these numbers?
</details>

<details><summary>Hint 3</summary>

`find_peaks` returns sample positions, not times.
</details>

**Check** (don't edit):

In [ ]:
current = 6
results[6] = {}
try:
    check(np.allclose(rr, np.diff(peaks) / fs), "RR intervals are in seconds")
    check(abs(hr_bpm - 72) < 1, "heart rate close to the true 72 bpm")
except NameError:
    results[6] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Explain your fix** (double-click to edit):

- The wrong assumption was: …
- The evidence that showed it: …
- A check that would catch it again: …

## Challenge 7 — A close-up of the first second ★★

**Goal:** Make `first_second`: the first second of the recording, centred on zero (mean removed), and plot it — **without changing the recording itself**.

**Symptom:** Later analysis of the full recording gives different numbers than before the close-up was made.

In [ ]:
ecg7 = make_ecg(seed=6)
first_second = ecg7[:fs]
first_second -= first_second.mean()                   # centre it for the close-up plot
t7 = np.arange(fs) / fs
plt.figure(figsize=(6, 2)); plt.plot(t7, first_second)
plt.xlabel("time (s)"); plt.ylabel("amplitude (mV)"); plt.show()

**Stuck?** Open one hint at a time.

<details><summary>Hint 1</summary>

Compare `ecg7` with a fresh `make_ecg(seed=6)`. Where do they differ?
</details>

<details><summary>Hint 2</summary>

`np.flatnonzero(ecg7 != make_ecg(seed=6))` lists the changed samples. Which ones are they?
</details>

<details><summary>Hint 3</summary>

Is a slice of a NumPy array a new array, or a window onto the old one?
</details>

**Check** (don't edit):

In [ ]:
current = 7
results[7] = {}
try:
    original = make_ecg(seed=6)
    check(np.array_equal(ecg7, original), "the full recording is unchanged")
    check(np.size(first_second) == fs and np.allclose(first_second, original[:fs] - original[:fs].mean()),
          "first_second is the first second, centred on zero")
except NameError:
    results[7] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Explain your fix** (double-click to edit):

- The wrong assumption was: …
- The evidence that showed it: …
- A check that would catch it again: …

## Challenge 8 — Downsampling for storage ★★★

**Goal:** A 10-s recording sampled at 500 Hz contains a 5 Hz rhythm (amplitude 1) and 60 Hz mains interference. Downsample it to 100 Hz (1000 samples) to save space, keeping the 5 Hz rhythm.

**Symptom:** The downsampled spectrum has a peak at 40 Hz that wasn't in the original.

**Predict first:** how many samples should the result have, and which frequencies should its spectrum show?

In [ ]:
fs8 = 500
t8 = np.arange(10 * fs8) / fs8
x8 = np.sin(2 * np.pi * 5 * t8) + 0.5 * np.sin(2 * np.pi * 60 * t8)
x8_100 = x8[::5]                                      # keep every 5th sample: 500 Hz -> 100 Hz

f8, P8 = signal.welch(x8_100, fs=100, nperseg=200)
plt.figure(figsize=(6, 2.5)); plt.semilogy(f8, P8)
plt.xlabel("frequency (Hz)"); plt.ylabel("PSD (a.u.²/Hz)"); plt.show()

**Stuck?** Open one hint at a time.

<details><summary>Hint 1</summary>

What is the highest frequency a 100 Hz recording can represent?
</details>

<details><summary>Hint 2</summary>

Where can a 60 Hz component go when it can't be represented? (100 − 60 = ?)
</details>

<details><summary>Hint 3</summary>

Something has to remove the 60 Hz *before* samples are thrown away.
</details>

**Check** (don't edit):

In [ ]:
current = 8
results[8] = {}
try:
    fc, Pc = signal.welch(np.asarray(x8_100), fs=100, nperseg=200)
    band = lambda lo, hi: Pc[(fc >= lo) & (fc <= hi)].sum() * (fc[1] - fc[0])          # power in a band
    check(np.size(x8_100) == 1000, "1000 samples: 10 s at 100 Hz")
    check(band(38, 42) < 1e-3, "no 40 Hz alias")
    check(0.4 < band(4, 6) < 0.6, "the 5 Hz rhythm keeps its power (0.5 for amplitude 1)")
except NameError:
    results[8] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Explain your fix** (double-click to edit):

- The wrong assumption was: …
- The evidence that showed it: …
- A check that would catch it again: …

## Challenge 9 — A classifier that is too good ★★★

**Goal:** Estimate how well a classifier predicts the label for a new subject. The subject-wise splits `splits9` are given and correct. The features are random noise and the labels are random, so the **expected** accuracy is 50 %; one finite dataset can land a little above or below by chance. Keep the classifier in a variable called `model9`.

**Symptom:** It reports about 80 % accuracy on pure noise.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.pipeline import Pipeline, make_pipeline

rng9 = np.random.default_rng(2013)
X9 = rng9.standard_normal((120, 500))               # 120 epochs x 500 noise features
y9 = rng9.integers(0, 2, 120)                        # random labels
groups9 = np.repeat(np.arange(12), 10)              # 12 subjects, 10 epochs each
splits9 = list(GroupKFold(4).split(X9, y9, groups9))   # given, and correct: by subject

X9_sel = SelectKBest(f_classif, k=10).fit_transform(X9, y9)       # keep the 10 best features
model9 = LogisticRegression(max_iter=1000)
acc9 = cross_val_score(model9, X9_sel, y9, cv=splits9).mean()
print(f"accuracy: {acc9:.2f}")

**Stuck?** Open one hint at a time.

<details><summary>Hint 1</summary>

A score far above what is possible deserves suspicion. Which steps here *learn* from data?
</details>

<details><summary>Hint 2</summary>

`SelectKBest` uses the labels to choose features. Which labels — the training fold's, or all of them?
</details>

<details><summary>Hint 3</summary>

A `Pipeline` re-fits every step inside each training fold.
</details>

**Check** (don't edit):

In [ ]:
current = 9
results[9] = {}
try:
    steps9 = [type(s).__name__ for _, s in model9.steps] if isinstance(model9, Pipeline) else []
    check("SelectKBest" in steps9, "feature selection is a step of model9, so it is re-fitted inside every fold")
    check("SelectKBest" in steps9 and np.isclose(cross_val_score(model9, X9, y9, cv=splits9).mean(), acc9),
          "acc9 comes from model9 on the raw features with the given splits")
    check(0.3 < acc9 < 0.7, "accuracy close to the expected 50 % (a diagnostic, not the proof)")
except NameError:
    results[9] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Explain your fix** (double-click to edit):

- The wrong assumption was: …
- The evidence that showed it: …
- A check that would catch it again: …

## Challenge 10 — The wrong split unit ★★★

**Goal:** Each of 30 patients either has a condition or not (one label per patient), and contributes 10 epochs. Estimate how well a model recognises the condition in a **new patient**. Keep the splits you use in a variable called `splits10`.

**Symptom:** It reports about 96 % accuracy — better than seems realistic for this weak signal.

**Predict first:** should recognising the condition in a *new* patient be easier or harder than in a patient the model has already seen?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, KFold, GroupKFold

rng10 = np.random.default_rng(2013)
patients10 = np.repeat(np.arange(30), 10)                         # 30 patients x 10 epochs
y10 = np.array([0, 1] * 15)[patients10]                           # one label per patient
fingerprint = rng10.normal(0, 1.0, (30, 6))                       # every patient's signals look a bit different
X10 = fingerprint[patients10] + rng10.normal(0, 0.3, (300, 6))
X10[:, 0] += 1.5 * y10                                            # the real, weaker effect of the condition

model10 = RandomForestClassifier(n_estimators=200, random_state=0)
splits10 = list(KFold(5, shuffle=True, random_state=0).split(X10))
acc10 = cross_val_score(model10, X10, y10, cv=splits10).mean()
print(f"accuracy: {acc10:.2f}")

**Stuck?** Open one hint at a time.

<details><summary>Hint 1</summary>

What is the question: new *epochs* of known patients, or new *patients*?
</details>

<details><summary>Hint 2</summary>

For one fold, print the patients in the training part and in the test part. Do they overlap?
</details>

<details><summary>Hint 3</summary>

The splitter needs to know which epochs belong to which patient.
</details>

**Check** (don't edit):

In [ ]:
current = 10
results[10] = {}
try:
    disjoint = all(set(patients10[tr]).isdisjoint(patients10[te]) for tr, te in splits10)
    covered = sorted(np.concatenate([te for _, te in splits10]).tolist()) == list(range(len(y10)))
    check(disjoint, "no patient is in both the training and the test part of any fold")
    check(covered, "every epoch is tested exactly once")
    check(disjoint and np.isclose(cross_val_score(model10, X10, y10, cv=splits10).mean(), acc10),
          "acc10 comes from model10 with these splits")
except NameError:
    results[10] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Explain your fix** (double-click to edit):

- The wrong assumption was: …
- The evidence that showed it: …
- A check that would catch it again: …

**Think further (optional).** There are 300 epochs but only 30 patients. Which number tells you how much
independent evidence you have about *new patients*? Try averaging the predictions per patient and scoring one
prediction per patient: how does that compare with the epoch-level accuracy?

There is no single "right" score here: 0.68 is what this split, seed and model give. Leave-one-patient-out or
another grouping is just as valid, as long as no patient is on both sides of a split.

## Done?

Run the cell below for a summary of all checks. When everything passes, compare *how* you found each bug with
the [worked troubleshooting steps](09_debugging_challenge_solutions.ipynb) — the route matters more than the fix. Then go back to the
[overview](README.md).

In [ ]:
EXPECTED = {1: 1, 2: 3, 3: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 3, 9: 3, 10: 3}     # number of checks per challenge
total = sum(EXPECTED.values())
passed = 0
for i, n in EXPECTED.items():
    r = results.get(i, "not run")
    if r == "not run":
        status = "not checked yet"
    elif r is None:
        status = "code didn't finish"
    else:
        passed += sum(r.values())
        status = f"{sum(r.values())} of {n} pass"
    print(f"Challenge {i:2d}: {status}")
print(f"\n{passed} of {total} checks pass.")
print("This reflects the last time each check cell ran. After editing an answer, re-run its check cell.")